# Experimento unimodal audio SIN VAD

In [ ]:
import torch
import os
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm
from huggingface_hub import login

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, recall_score, f1_score, precision_score
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import GridSearchCV


RANDOM_STATE = 123

/home/ethe/miniconda3/envs/tfg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from collections import defaultdict
from sklearn.decomposition import PCA

def evaluate_embeddings(
    filepath,
    feature_regex,
    target_col="binary-stress",
    group_col="subject",
    task_col="subject_activity",
    outer_splits=5,
    inner_splits=3,
    random_state=123,
    pca_components=None
):
    df = pd.read_csv(filepath)

    X = df.filter(regex=feature_regex)
    y = df[target_col]
    groups = df[group_col]
    task_ids = df[task_col]

    sgkf     = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    cv_inner = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

    agg_strategies = {
        "Mean": "mean",
        "Median": "median",
        "Max": "max",
        "75th_Percentile": lambda x: np.percentile(x, 75)
    }

    metric_keys = ["Bal_Acc", "Accuracy", "F1", "Rec_Relaxed", "Rec_Stress", "Precision"]
    fold_results = defaultdict(list)


    def make_pca_step(with_scaler=False):
        steps = []
        if pca_components is not None:
            if with_scaler:
                steps.append(("scaler", StandardScaler()))
            steps.append(("pca", PCA(n_components=pca_components, random_state=random_state)))
        return steps


    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):

        assert not (set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), \
            f"Fold {fold_idx}: subject leakage detected"

        X_train, X_test = X.iloc[train_idx],  X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx],  y.iloc[test_idx]
        groups_train    = groups.iloc[train_idx]
        task_ids_train  = task_ids.iloc[train_idx]
        task_ids_test   = task_ids.iloc[test_idx]
        scale_weight    = (y_train == 0).sum() / (y_train == 1).sum()

        models = {
            "Logistic Regression": Pipeline([
                ("scaler", StandardScaler()),
                *make_pca_step(),
                ("clf", LogisticRegression(class_weight='balanced', max_iter=2000, random_state=random_state))
            ]),
            "Balanced Random Forest": Pipeline([
                *make_pca_step(with_scaler=True),
                ("clf", BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto',
                                                       random_state=random_state, n_jobs=1))
            ]),
            "XGBoost": Pipeline([
                *make_pca_step(with_scaler=True),
                ("clf", XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss',
                                      random_state=random_state, n_jobs=1))
            ]),
            "Random Forest": Pipeline([
                *make_pca_step(with_scaler=True),
                ("clf", RandomForestClassifier(class_weight="balanced", n_estimators=100,
                                               random_state=random_state, n_jobs=1))
            ]),
        }

        for model_name, model in models.items():

            model.fit(X_train, y_train)
            test_probs = model.predict_proba(X_test)[:, 1]

            oof_probs = cross_val_predict(
                model, X_train, y_train,
                groups=groups_train,
                cv=cv_inner,
                method='predict_proba',
                n_jobs=-1
            )[:, 1]

            train_base_df = pd.DataFrame({
                "subject_activity": task_ids_train.values,
                "true_label":       y_train.values,
                "pred_prob":        oof_probs
            })
            test_base_df = pd.DataFrame({
                "subject_activity": task_ids_test.values,
                "true_label":       y_test.values,
                "pred_prob":        test_probs
            })

            for strategy_name, agg_func in agg_strategies.items():

                train_agg = train_base_df.groupby("subject_activity").agg(
                    true_label=("true_label", "first"),
                    agg_prob=("pred_prob", agg_func)
                ).reset_index()

                best_thresh, best_b_acc = 0.5, 0.0
                for thresh in np.arange(0.1, 0.9, 0.02):
                    preds = (train_agg["agg_prob"] >= thresh).astype(int)
                    b_acc = balanced_accuracy_score(train_agg["true_label"], preds)
                    if b_acc >= best_b_acc:
                        best_b_acc = b_acc
                        best_thresh = thresh

                test_agg = test_base_df.groupby("subject_activity").agg(
                    true_label=("true_label", "first"),
                    agg_prob=("pred_prob", agg_func)
                ).reset_index()

                test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)

                fold_results[(model_name, strategy_name)].append({
                    "Thresh":      best_thresh,
                    "Bal_Acc":     balanced_accuracy_score(test_agg["true_label"], test_preds),
                    "Accuracy":    accuracy_score(test_agg["true_label"], test_preds),
                    "F1":          f1_score(test_agg["true_label"], test_preds, zero_division=0),
                    "Rec_Relaxed": recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0),
                    "Rec_Stress":  recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0),
                    "Precision":   precision_score(test_agg["true_label"], test_preds, zero_division=0),
                })

    # --- Aggregate across folds ---
    summary_rows = []
    for (model_name, strategy_name), fold_metrics in fold_results.items():
        row = {"Model": model_name, "Strategy": strategy_name}
        for key in ["Thresh"] + metric_keys:
            values = [m[key] for m in fold_metrics]
            row[f"{key}_mean"] = round(float(np.mean(values)), 4)
            row[f"{key}_std"]  = round(float(np.std(values)),  4)
        summary_rows.append(row)

    summary_df = (
        pd.DataFrame(summary_rows)
        .sort_values("Bal_Acc_mean", ascending=False)
        .reset_index(drop=True)
    )

    # --- Format display ---
    display_df = summary_df[["Model", "Strategy"]].copy()
    display_df.insert(2, "Thresh", summary_df["Thresh_mean"].map("{:.2f}".format))
    for key in metric_keys:
        display_df[key] = (
            summary_df[f"{key}_mean"].map("{:.4f}".format)
            + " ± "
            + summary_df[f"{key}_std"].map("{:.4f}".format)
        )

    embedding_name = filepath.split("/")[-1].replace(".csv", "")
    pca_tag = f", PCA={pca_components}" if pca_components is not None else ""
    print(f"\n=== Results: {embedding_name} ({outer_splits}-Fold CV{pca_tag}, mean ± std) ===")
    print(display_df.to_string(index=False))

In [ ]:

def evaluate_best_model_grid_search(
    filepath,
    feature_regex,
    model_name,           # e.g. "XGBoost"
    aggregation_strategy, # e.g. "75th_Percentile"
    param_grid,           # dict of hyperparameters to search
    target_col="binary-stress",
    group_col="subject",
    task_col="subject_activity",
    outer_splits=5,
    inner_splits=3,
    random_state=123,
    pca_components=None
):
    df = pd.read_csv(filepath)

    X = df.filter(regex=feature_regex)
    y = df[target_col]
    groups = df[group_col]
    task_ids = df[task_col]

    sgkf = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    cv_inner = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

    agg_strategies = {
        "Mean": "mean",
        "Median": "median",
        "Max": "max",
        "75th_Percentile": lambda x: np.percentile(x, 75)
    }
    agg_func = agg_strategies[aggregation_strategy]

    metric_keys  = ["Bal_Acc", "Accuracy", "F1", "Rec_Relaxed", "Rec_Stress", "Precision"]
    fold_results = []

    def make_pca_step(with_scaler=False):
        steps = []
        if pca_components is not None:
            if with_scaler:
                steps.append(("scaler", StandardScaler()))
            steps.append(("pca", PCA(n_components=pca_components, random_state=random_state)))
        return steps

    def build_base_model(scale_weight=None):
        if model_name == "Logistic Regression":
            return Pipeline([
                ("scaler", StandardScaler()),
                *make_pca_step(),
                ("clf", LogisticRegression(class_weight='balanced', max_iter=2000, random_state=random_state))
            ])
        elif model_name == "Balanced Random Forest":
            return Pipeline([
                ("scaler", StandardScaler()),
                *make_pca_step(),
                ("clf", BalancedRandomForestClassifier(sampling_strategy='auto', random_state=random_state, n_jobs=1))
            ])
        elif model_name == "XGBoost":
            return Pipeline([
                ("scaler", StandardScaler()),
                *make_pca_step(),
                ("clf", XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss',
                                      random_state=random_state, n_jobs=1))
            ])
        elif model_name == "Random Forest":
            return Pipeline([
                ("scaler", StandardScaler()),
                *make_pca_step(),
                ("clf", RandomForestClassifier(class_weight="balanced", random_state=random_state, n_jobs=1))
            ])
        else:
            raise ValueError(f"Unknown model: {model_name}")

    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):

        assert not (set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), \
            f"Fold {fold_idx}: subject leakage detected"

        X_train, X_test = X.iloc[train_idx],  X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx],  y.iloc[test_idx]
        groups_train    = groups.iloc[train_idx]
        task_ids_train  = task_ids.iloc[train_idx]
        task_ids_test   = task_ids.iloc[test_idx]
        scale_weight    = (y_train == 0).sum() / (y_train == 1).sum()

        base_model = build_base_model(scale_weight=scale_weight)

        # Grid search using inner CV — no test data touches this
        search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            scoring="balanced_accuracy",
            cv=cv_inner,
            n_jobs=-1,
            refit=True
        )
        search.fit(X_train, y_train, groups=groups_train)
        best_model = search.best_estimator_

        print(f"Fold {fold_idx} best params: {search.best_params_}")

        # OOF probs for threshold tuning using the best configuration
        oof_probs = cross_val_predict(
            best_model, X_train, y_train,
            groups=groups_train,
            cv=cv_inner,
            method='predict_proba',
            n_jobs=-1
        )[:, 1]

        # Refit best model on full training fold, predict on test
        best_model.fit(X_train, y_train)
        test_probs = best_model.predict_proba(X_test)[:, 1]

        # Aggregate and tune threshold on OOF
        train_base_df = pd.DataFrame({
            "subject_activity": task_ids_train.values,
            "true_label":       y_train.values,
            "pred_prob":        oof_probs
        })
        train_agg = train_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        best_thresh, best_b_acc = 0.5, 0.0
        for thresh in np.arange(0.1, 0.9, 0.02):
            preds = (train_agg["agg_prob"] >= thresh).astype(int)
            b_acc = balanced_accuracy_score(train_agg["true_label"], preds)
            if b_acc >= best_b_acc:
                best_b_acc = b_acc
                best_thresh = thresh

        # Evaluate on test fold
        test_base_df = pd.DataFrame({
            "subject_activity": task_ids_test.values,
            "true_label":       y_test.values,
            "pred_prob":        test_probs
        })
        test_agg = test_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)

        fold_results.append({
            "Fold":        fold_idx,
            "Best_Params": search.best_params_,
            "Thresh":      round(best_thresh, 2),
            "Bal_Acc":     balanced_accuracy_score(test_agg["true_label"], test_preds),
            "Accuracy":    accuracy_score(test_agg["true_label"], test_preds),
            "F1":          f1_score(test_agg["true_label"], test_preds, zero_division=0),
            "Rec_Relaxed": recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0),
            "Rec_Stress":  recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0),
            "Precision":   precision_score(test_agg["true_label"], test_preds, zero_division=0),
        })

    # Summary
    results_df = pd.DataFrame(fold_results)
    print(f"\n=== {model_name} + GridSearch ({aggregation_strategy}, {outer_splits}-Fold CV) ===")
    print(results_df.to_string(index=False))

    print(f"\n--- Mean ± Std ---")
    for key in metric_keys:
        mean = results_df[key].mean()
        std  = results_df[key].std()
        print(f"{key}: {mean:.4f} ± {std:.4f}")

    return results_df

# Windowing

In [8]:
import os
import numpy as np
import soundfile as sf
import librosa

def save_audio_windows(input_dir, output_dir, window_size=5.0, overlap=0.0, sampling_rate=16000):
    """
    Splits each audio file into fixed-length windows and saves them.

    input_dir structure:
        input_dir/subject/activity.wav

    output_dir structure:
        output_dir/subject/activity/window_0.wav
    """

    stride = window_size * (1 - overlap)

    for subject in sorted(os.listdir(input_dir)):
        subject_dir = os.path.join(input_dir, subject)

        if not os.path.isdir(subject_dir):
            continue

        for act_file in sorted(os.listdir(subject_dir)):
            if not act_file.lower().endswith(".wav"):
                continue

            audio_path = os.path.join(subject_dir, act_file)
            activity = os.path.splitext(act_file)[0]

            y, sr = librosa.load(audio_path, sr=sampling_rate)
            duration = librosa.get_duration(y=y, sr=sr)

            out_act_dir = os.path.join(output_dir, subject, activity)
            os.makedirs(out_act_dir, exist_ok=True)

            start = 0.0
            window_id = 0

            while start + window_size <= duration:
                end = start + window_size

                start_sample = int(start * sr)
                end_sample = int(end * sr)

                y_window = y[start_sample:end_sample]

                out_path = os.path.join(out_act_dir, f"window_{window_id}.wav")
                sf.write(out_path, y_window, sr)

                window_id += 1
                start += stride

            if window_id == 0:
                print(f"No complete windows for: {audio_path}")

        print(f"{subject} done.")

    print("All audio windows saved.")

In [ ]:
save_audio_windows(input_dir='./data/original', output_dir='./data/5s_0overlap_original')

2ea4 done.
2hpu done.
2z7d done.
45lx done.
4e8r done.
4woj done.
5f7t done.
6g6y done.
6k5f done.
71i5 done.
7h5u done.
7m3c done.
No complete windows for: ./data/vad_applied/8g4y/8g4y_Math.wav
No complete windows for: ./data/vad_applied/8g4y/8g4y_Stroop.wav
8g4y done.
No complete windows for: ./data/vad_applied/8i4i/8i4i_Counting3.wav
No complete windows for: ./data/vad_applied/8i4i/8i4i_Math.wav
No complete windows for: ./data/vad_applied/8i4i/8i4i_Stroop.wav
8i4i done.
9j3o done.
No complete windows for: ./data/vad_applied/9t6n/9t6n_Counting2.wav
9t6n done.
9txq done.
a1k9 done.
b2l8 done.
b9w0 done.
bfl5 done.
c3m7 done.
chdf done.
ctzy done.
cxj0 done.
No complete windows for: ./data/vad_applied/d4n6/d4n6_Stroop.wav
d4n6 done.
e5p4 done.
g7r2 done.
g9j5 done.
No complete windows for: ./data/vad_applied/h7j3/h7j3_Counting2.wav
h7j3 done.
h8r2 done.
h8s1 done.
i9t9 done.
iqyg done.
No complete windows for: ./data/vad_applied/j9h8/j9h8_Counting3.wav
j9h8 done.
k2v7 done.
k67g done.


# Paralinguistic audio features extraction using LIBROSA:

In [4]:
import pandas as pd

def extract_features_from_audio(y, sr):
    features = {}

    # Energy / loudness
    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # Pitch / F0
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7")
    )

    valid_f0 = f0[~np.isnan(f0)]

    if len(valid_f0) > 0:
        features["pitch_mean"] = np.mean(valid_f0)
        features["pitch_std"] = np.std(valid_f0)
        features["pitch_min"] = np.min(valid_f0)
        features["pitch_max"] = np.max(valid_f0)
    else:
        features["pitch_mean"] = 0.0
        features["pitch_std"] = 0.0
        features["pitch_min"] = 0.0
        features["pitch_max"] = 0.0

    # Voiced ratio
    features["voiced_ratio"] = np.mean(voiced_flag) if voiced_flag is not None else 0.0

    # Pause / low-energy ratio
    if len(rms) > 0:
        silence_threshold = np.percentile(rms, 25)
        features["low_energy_ratio"] = np.mean(rms < silence_threshold)
    else:
        features["low_energy_ratio"] = 0.0

    return features

def extract_audio_window_features(windows_dir, output_csv, sampling_rate=16000):

    rows = []

    for subject in sorted(os.listdir(windows_dir)):
        subject_dir = os.path.join(windows_dir, subject)

        if not os.path.isdir(subject_dir):
            continue

        for activity in sorted(os.listdir(subject_dir)):
            activity_dir = os.path.join(subject_dir, activity)

            if not os.path.isdir(activity_dir):
                continue

            for window_file in sorted(os.listdir(activity_dir)):
                if not window_file.lower().endswith(".wav"):
                    continue

                window_path = os.path.join(activity_dir, window_file)

                window_id = int(
                    os.path.splitext(window_file)[0].replace("window_", "")
                )

                y, sr = librosa.load(window_path, sr=sampling_rate)

                features = extract_features_from_audio(y, sr)

                rows.append({
                    "subject": subject,
                    "activity": activity.split('_')[1],
                    "window_id": window_id,
                    "subject_activity": activity,
                    **features
                })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    print(f"Saved features to: {output_csv}")
    print(f"Total windows processed: {len(df)}")

    return df

In [ ]:
audio_features = extract_audio_window_features(
    windows_dir="./data/5s_0overlap_original",
    output_csv="audio_features_5s_0overlap_original.csv",
    sampling_rate=16000
)

In [3]:
df_features = pd.read_csv('audio_features_5s_0overlap_original.csv')
df_features

,subject,activity,window_id,subject_activity,rms_mean,rms_std,zcr_mean,zcr_std,mfcc_1_mean,mfcc_1_std,...,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,pitch_mean,pitch_std,pitch_min,pitch_max,voiced_ratio,low_energy_ratio
0,2ea4,Counting1,0,2ea4_Counting1,0.014279,0.019869,0.112812,0.071869,-463.51035,163.373400,...,-9.523801,7.433795,-5.596460,8.518366,194.808267,24.568569,159.199616,244.105284,0.554140,0.248408
1,2ea4,Counting1,1,2ea4_Counting1,0.022294,0.020725,0.124107,0.075209,-372.64136,152.708270,...,-6.754907,8.368198,-6.083131,9.427306,156.321563,6.498795,144.309891,163.864521,0.636943,0.248408
2,2ea4,Counting1,10,2ea4_Counting1,0.015378,0.024722,0.132797,0.089800,-458.43506,162.470400,...,-4.263349,5.551752,-3.264464,5.581558,154.927750,42.533571,68.896544,186.068889,0.261146,0.248408
3,2ea4,Counting1,11,2ea4_Counting1,0.003856,0.010959,0.117281,0.038330,-572.58795,91.507980,...,-5.842841,4.904461,-4.305291,6.688785,163.697410,1.270558,162.920731,166.728822,0.070064,0.248408
4,2ea4,Counting1,2,2ea4_Counting1,0.019461,0.028323,0.130505,0.062242,-411.32623,152.807140,...,-2.316386,8.522891,-5.307195,6.115133,159.980194,9.141810,136.210400,179.730700,0.363057,0.248408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,y9z6,Stroop,5,y9z6_Stroop,0.003811,0.003413,0.155410,0.061668,-430.51596,15.959509,...,2.503946,4.481340,-3.208373,4.202956,251.270476,84.825726,73.416192,329.627557,0.541401,0.248408
4532,y9z6,Stroop,6,y9z6_Stroop,0.005182,0.003681,0.171763,0.139788,-417.96900,27.945390,...,4.778991,5.462026,-1.967502,5.877249,294.420870,14.757685,272.420799,329.627557,0.471338,0.248408
4533,y9z6,Stroop,7,y9z6_Stroop,0.004975,0.003693,0.199670,0.145555,-415.02948,27.261093,...,3.933442,6.173757,-1.899312,4.486299,284.000813,9.722752,267.740773,314.742105,0.477707,0.248408
4534,y9z6,Stroop,8,y9z6_Stroop,0.004666,0.003151,0.174926,0.108746,-413.44302,23.451452,...,1.862005,5.193305,-1.741245,4.676744,292.071322,21.131181,249.810974,329.627557,0.477707,0.248408


# Task level aggregation:

1. Feature level aggregation:

In [4]:
def task_aggregation(df_audio, gt):
    df = df_audio.copy()

    id_cols = ["subject", "activity", "window_id", "subject_activity"]
    feature_cols = [c for c in df.columns if c not in id_cols]

    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Aggregate window-level features to task level while keeping the subject ID.
    # This lets us split by subject, so no subject appears in both train and test.
    agg = df.groupby(["subject_activity", "subject"])[feature_cols].agg(["mean", "std", "max"])
    agg.columns = [f"{feat}_{stat}" for feat, stat in agg.columns]
    agg = agg.reset_index()

    # Merge with GT
    dataset = gt.merge(agg, on="subject_activity", how="inner")

    groups = dataset["subject"]
    X = dataset.drop(columns=["subject", "subject_activity", "y_true"])
    y = dataset["y_true"].astype(int)

    return X, y, groups, dataset

gt = pd.read_csv('../labels_v2.csv', sep=';').rename(columns={'subject/task':'subject_activity', 'binary-stress':'y_true'}).drop(['affect3-class', 'affect3-class-v2'], axis=1)
gt


,subject_activity,y_true
0,2ea4_Breathing,0
1,2ea4_Counting1,1
2,2ea4_Counting2,1
3,2ea4_Counting3,1
4,2ea4_Math,1
...,...,...
695,y9z6_Relax,0
696,y9z6_Speaking,1
697,y9z6_Stroop,1
698,y9z6_Video1,1


In [5]:
X, y, groups, agg_df = task_aggregation(df_features, gt)

In [30]:
X.isna().sum()

rms_mean_mean             0
rms_mean_std             10
rms_mean_max              0
rms_std_mean              0
rms_std_std              10
                         ..
voiced_ratio_std         10
voiced_ratio_max          0
low_energy_ratio_mean     0
low_energy_ratio_std     10
low_energy_ratio_max      0
Length: 108, dtype: int64

In [31]:
nan_cols = agg_df.isna().sum()
nan_cols[nan_cols > 0].sort_values(ascending=False)

rms_mean_std            10
rms_std_std             10
zcr_mean_std            10
zcr_std_std             10
mfcc_1_mean_std         10
mfcc_1_std_std          10
mfcc_2_mean_std         10
mfcc_2_std_std          10
mfcc_3_mean_std         10
mfcc_3_std_std          10
mfcc_4_mean_std         10
mfcc_4_std_std          10
mfcc_5_mean_std         10
mfcc_5_std_std          10
mfcc_6_mean_std         10
mfcc_6_std_std          10
mfcc_7_mean_std         10
mfcc_7_std_std          10
mfcc_8_mean_std         10
mfcc_8_std_std          10
mfcc_9_mean_std         10
mfcc_9_std_std          10
mfcc_10_mean_std        10
mfcc_10_std_std         10
mfcc_11_mean_std        10
mfcc_11_std_std         10
mfcc_12_mean_std        10
mfcc_12_std_std         10
mfcc_13_mean_std        10
mfcc_13_std_std         10
pitch_mean_std          10
pitch_std_std           10
pitch_min_std           10
pitch_max_std           10
voiced_ratio_std        10
low_energy_ratio_std    10
dtype: int64

Estos son los casos en que solo se pudo extraer 1 ventana, por lo que no se pudo computar la desviacion. Podemos imputarlos con 0, refiriendo a que no hay cambios entre ventanas.

In [5]:
std_cols = [c for c in agg_df.columns if c.endswith("_std")]
agg_df[std_cols] = agg_df[std_cols].fillna(0)
X[std_cols] = X[std_cols].fillna(0)
nan_cols = agg_df.isna().sum()
nan_cols[nan_cols > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [6]:
y.value_counts(normalize=True)

y_true
1    0.708995
0    0.291005
Name: proportion, dtype: float64

In [7]:
gt

,subject_activity,y_true
0,2ea4_Breathing,0
1,2ea4_Counting1,1
2,2ea4_Counting2,1
3,2ea4_Counting3,1
4,2ea4_Math,1
...,...,...
695,y9z6_Relax,0
696,y9z6_Speaking,1
697,y9z6_Stroop,1
698,y9z6_Video1,1


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, precision_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict


def task_aggregation(df_audio, gt):
    df = df_audio.copy()

    id_cols = ["subject", "activity", "window_id", "subject_activity"]
    feature_cols = [c for c in df.columns if c not in id_cols]

    # Ensure features are numeric
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Group by BOTH subject_activity and subject to keep the subject ID alive
    # The split uses subject, not subject_activity, to avoid subject leakage
    agg = df.groupby(["subject_activity", "subject"])[feature_cols].agg(["mean", "std", "max"])
    agg.columns = [f"{feat}_{stat}" for feat, stat in agg.columns]
    agg = agg.reset_index()

    # Merge with Ground Truth
    dataset = gt.merge(agg, on="subject_activity", how="inner")

    # Subject-aware cross-validation groups
    groups = dataset["subject"]

    # Drop all non-feature cols
    X = dataset.drop(columns=["subject", "subject_activity", "y_true"])
    X = X.fillna(0)

    y = dataset["y_true"].astype(int)

    return X, y, groups, dataset


df_audio = pd.read_csv("./audio_features_5s_0overlap_original.csv")
gt = pd.read_csv('../labels_v2.csv', sep=';').rename(columns={'subject/task':'subject_activity', 'binary-stress':'y_true'}).drop(['affect3-class', 'affect3-class-v2'], axis=1)

X_agg, y_agg, groups_agg, full_dataset = task_aggregation(df_audio, gt)
task_ids_agg = full_dataset["subject_activity"]

sgkf_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf_split.split(X_agg, y_agg, groups=groups_agg))

# Guardrail: fail fast if any subject appears in both train and test.
train_subjects = set(groups_agg.iloc[train_idx])
test_subjects = set(groups_agg.iloc[test_idx])
overlap = train_subjects & test_subjects
assert not overlap, f"Subject leakage detected: {sorted(overlap)}"

X_train = X_agg.iloc[train_idx]
y_train = y_agg.iloc[train_idx]
groups_train = groups_agg.iloc[train_idx]
task_ids_train = task_ids_agg.iloc[train_idx]

X_test = X_agg.iloc[test_idx]
y_test = y_agg.iloc[test_idx]
groups_test = groups_agg.iloc[test_idx]
task_ids_test = task_ids_agg.iloc[test_idx]

# MODELS
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Balanced Random Forest": make_pipeline(
        BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

cv_train = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
results_list = []

for model_name, model in models.items():

    # Out-of-fold probabilities for threshold tuning using subject-grouped CV
    train_unbiased_probs = cross_val_predict(
        model, X_train, y_train,
        groups=groups_train,
        cv=cv_train,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    best_thresh = 0.5
    best_b_acc = 0

    for thresh in np.arange(0.1, 0.9, 0.02):
        train_preds = (train_unbiased_probs >= thresh).astype(int)
        b_acc = balanced_accuracy_score(y_train, train_preds)
        if b_acc >= best_b_acc:
            best_b_acc = b_acc
            best_thresh = thresh

    # Train final model on the full subject-split training data
    model.fit(X_train, y_train)

    # Predict on the held-out subjects
    test_probs = model.predict_proba(X_test)[:, 1]
    test_preds = (test_probs >= best_thresh).astype(int)

    final_b_acc = balanced_accuracy_score(y_test, test_preds)
    acc = accuracy_score(y_test, test_preds)
    f1 = f1_score(y_test, test_preds)
    recall_0 = recall_score(y_test, test_preds, pos_label=0, zero_division=0)
    recall_1 = recall_score(y_test, test_preds, pos_label=1, zero_division=0)
    precision = precision_score(y_test, test_preds, zero_division=0)

    results_list.append({
        "Model": model_name,
        "Thresh": round(best_thresh, 2),
        "Bal_Acc": round(final_b_acc, 4),
        "Accuracy": round(acc, 2),
        "F1": round(f1, 2),
        "Rec_Relaxed": round(recall_0, 2),
        "Rec_Stress": round(recall_1, 2),
        "Precision": round(precision, 2)
    })

comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by="Bal_Acc", ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))


                 Model  Thresh  Bal_Acc  Accuracy   F1  Rec_Relaxed  Rec_Stress  Precision
               XGBoost    0.82   0.4524      0.51 0.63         0.33        0.57       0.70
   Logistic Regression    0.50   0.3869      0.45 0.59         0.24        0.54       0.65
Balanced Random Forest    0.48   0.3423      0.39 0.52         0.24        0.45       0.61
         Random Forest    0.70   0.3423      0.39 0.52         0.24        0.45       0.61


## Decision level Agregation methods tuning

In [6]:
features_df = pd.read_csv("./audio_features_5s_0overlap_original.csv")
gt_df = pd.read_csv("../labels_v2.csv", sep=";")

merged_df = pd.merge(
    features_df,
    gt_df,
    left_on="subject_activity",
    right_on="subject/task",
    how="inner"
)

# feature columns and target
exclude_cols = [
    "subject", "activity", "window_id", "subject_activity",
    "subject/task", "binary-stress", "affect3-class", "affect3-class-v2"
]
feature_cols = [col for col in merged_df.columns if col not in exclude_cols]

X = merged_df[feature_cols]
y = merged_df["binary-stress"]

# Split groups must be subjects, not tasks. This prevents different tasks from the
# same subject from ending up in both train and test.
groups = merged_df["subject"]
task_ids = merged_df["subject_activity"]

# Stratified Group Splitting
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# next() to grab just the first split
train_idx, test_idx = next(sgkf.split(X, y, groups))

train_subjects = set(groups.iloc[train_idx])
test_subjects = set(groups.iloc[test_idx])
overlap = train_subjects & test_subjects
assert not overlap, f"Subject leakage detected: {sorted(overlap)}"

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
task_ids_train, task_ids_test = task_ids.iloc[train_idx], task_ids.iloc[test_idx]


In [10]:
# Models
# # Note: XGBoost uses 'scale_pos_weight' for imbalance. Since Class 1 (Stress) 
# is the majority class, we calculate the ratio of Class 0 to Class 1.
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), 
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Balanced Random Forest": make_pipeline(
        BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "Random Forest":
        RandomForestClassifier(
            class_weight="balanced",
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
    ), 
}

agg_strategies = {
    "Mean": "mean",
    "Median": "median",
    "Max": "max",
    "75th_Percentile": lambda x: np.percentile(x, 75)
}

cv_train = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_list = []


for model_name, model in models.items():
    
    model.fit(X_train, y_train)
    test_probs = model.predict_proba(X_test)[:, 1]
    
    # Get out-of-fold probabilities for threshold tuning
    train_unbiased_probs = cross_val_predict(
        model, X_train, y_train, 
        groups=groups_train, 
        cv=cv_train, 
        method='predict_proba',
        n_jobs=-1
    )[:, 1]
    
    train_base_df = pd.DataFrame({
        "subject_activity": task_ids_train.values,
        "true_label": y_train.values,
        "pred_prob": train_unbiased_probs 
    })
    
    test_base_df = pd.DataFrame({
        "subject_activity": task_ids_test.values,
        "true_label": y_test.values,
        "pred_prob": test_probs 
    })

    for strategy_name, agg_func in agg_strategies.items():
        
        train_agg = train_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()
        
        # Tune Threshold
        best_thresh = 0.5
        best_b_acc = 0
        for thresh in np.arange(0.1, 0.9, 0.02):
            train_preds = (train_agg["agg_prob"] >= thresh).astype(int)
            b_acc = balanced_accuracy_score(train_agg["true_label"], train_preds)
            if b_acc >= best_b_acc: 
                best_b_acc = b_acc
                best_thresh = thresh
                
        test_agg = test_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()
        
        # Apply Threshold and Evaluate
        test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)
        final_b_acc = balanced_accuracy_score(test_agg["true_label"], test_preds)
        acc = accuracy_score(test_agg["true_label"], test_preds)
        f1 = f1_score(test_agg["true_label"], test_preds)
        recall_0 = recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0)
        recall_1 = recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0)
        precision = precision_score(test_agg["true_label"], test_preds)
        
        results_list.append({
            "Model": model_name,
            "Strategy": strategy_name,
            "Thresh": round(best_thresh, 2),
            "Bal_Acc": round(final_b_acc, 4),
            "Accuracy": round(acc,2),
            "F1": round(f1,2),
            "Rec_Relaxed": round(recall_0, 2),
            "Rec_Stress": round(recall_1, 2),
            "Precision": round(precision, 2)
        })

comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by="Bal_Acc", ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))

                 Model        Strategy  Thresh  Bal_Acc  Accuracy   F1  Rec_Relaxed  Rec_Stress  Precision
Balanced Random Forest             Max    0.70   0.4732      0.49 0.60         0.43        0.52       0.71
Balanced Random Forest          Median    0.54   0.4494      0.42 0.48         0.52        0.38       0.68
               XGBoost             Max    0.88   0.4375      0.64 0.78         0.00        0.88       0.70
         Random Forest             Max    0.86   0.4226      0.44 0.55         0.38        0.46       0.67
         Random Forest 75th_Percentile    0.80   0.4196      0.42 0.51         0.43        0.41       0.66
   Logistic Regression            Mean    0.58   0.4167      0.39 0.46         0.48        0.36       0.65
   Logistic Regression 75th_Percentile    0.62   0.4137      0.43 0.53         0.38        0.45       0.66
         Random Forest            Mean    0.74   0.4048      0.42 0.52         0.38        0.43       0.65
   Logistic Regression             Ma

# Audio Embeddings

## Wav2Vec 2.0:

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm
from huggingface_hub import login

AUDIO_WINDOWS_DIR = "./data/5s_0overlap_original"
GROUND_TRUTH_CSV = "../labels_v2.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# login() 
model_name = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device)
model.eval()

target_sample_rate = 16000
embedding_list = []

with torch.no_grad():
    for subject in tqdm(sorted(os.listdir(AUDIO_WINDOWS_DIR)), desc="Subjects"):
        subject_dir = os.path.join(AUDIO_WINDOWS_DIR, subject)
        if not os.path.isdir(subject_dir): continue
            
        for activity in sorted(os.listdir(subject_dir)):
            activity_dir = os.path.join(subject_dir, activity)
            if not os.path.isdir(activity_dir): continue
                
            subject_activity = f"{subject}_{activity}"
            
            for window_file in sorted(os.listdir(activity_dir)):
                if not window_file.lower().endswith(".wav"): continue
                    
                window_id = int(window_file.split("_")[1].split(".")[0])
                file_path = os.path.join(activity_dir, window_file)
                
                waveform, sample_rate = torchaudio.load(file_path)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                    
                if sample_rate != target_sample_rate:
                    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
                    waveform = resampler(waveform)
                    
                inputs = processor(waveform.squeeze().numpy(), sampling_rate=target_sample_rate, return_tensors="pt")
                inputs = {key: val.to(device) for key, val in inputs.items()}
                outputs = model(**inputs)
                
                mean_pooled_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                
                row_data = {
                    "subject": subject,
                    "activity": activity,
                    "subject_activity": subject_activity,
                    "window_id": window_id
                }
                
                for i, val in enumerate(mean_pooled_embedding):
                    row_data[f"w2v_{i}"] = val
                    
                embedding_list.append(row_data)

embeddings_df = pd.DataFrame(embedding_list)
embeddings_df['activity'] = embeddings_df['activity'].apply(lambda x: x.split('_')[1])
embeddings_df['subject_activity'] = embeddings_df['subject_activity'].apply(lambda x: "_".join(x.split('_')[1:3]))

gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";")

merged_df = pd.merge(
    embeddings_df, 
    gt_df, 
    left_on="subject_activity", 
    right_on="subject/task", 
    how="inner"
)

output_csv = "wav2vec_embeddings_original.csv"
merged_df.to_csv(output_csv, index=False)
print(f"Extraction complete! Saved {len(merged_df)} windows to {output_csv}.")

Using device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Subjects: 100%|██████████| 54/54 [02:05<00:00,  2.32s/it]


Extraction complete! Saved 4536 windows to wav2vec_embeddings_original.csv.


In [ ]:
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("wav2vec_embeddings_original.csv")

# Extract features, target, subject groups for splitting, and task IDs for aggregation
X = df.filter(regex='^w2v_')
y = df["binary-stress"]
groups = df["subject"]
task_ids = df["subject_activity"]

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups))

train_subjects = set(groups.iloc[train_idx])
test_subjects = set(groups.iloc[test_idx])
overlap = train_subjects & test_subjects
assert not overlap, f"Subject leakage detected: {sorted(overlap)}"

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
task_ids_train, task_ids_test = task_ids.iloc[train_idx], task_ids.iloc[test_idx]

# Calculate imbalance weight for XGBoost
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Define Models
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Balanced Random Forest": make_pipeline(
        BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        StandardScaler(),
        XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

agg_strategies = {
    "Mean": "mean",
    "Median": "median",
    "Max": "max",
    "75th_Percentile": lambda x: np.percentile(x, 75)
}

cv_train = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
results_list = []


for model_name, model in models.items():

    # Train and predict
    model.fit(X_train, y_train)
    test_probs = model.predict_proba(X_test)[:, 1]

    # Out-of-fold probabilities for unbiased threshold tuning, grouped by subject
    train_unbiased_probs = cross_val_predict(
        model, X_train, y_train,
        groups=groups_train,
        cv=cv_train,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    train_base_df = pd.DataFrame({
        "subject_activity": task_ids_train.values,
        "true_label": y_train.values,
        "pred_prob": train_unbiased_probs
    })

    test_base_df = pd.DataFrame({
        "subject_activity": task_ids_test.values,
        "true_label": y_test.values,
        "pred_prob": test_probs
    })

    # Test Aggregation Strategies
    for strategy_name, agg_func in agg_strategies.items():

        train_agg = train_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        best_thresh = 0.5
        best_b_acc = 0
        for thresh in np.arange(0.1, 0.9, 0.02):
            train_preds = (train_agg["agg_prob"] >= thresh).astype(int)
            b_acc = balanced_accuracy_score(train_agg["true_label"], train_preds)
            if b_acc >= best_b_acc:
                best_b_acc = b_acc
                best_thresh = thresh

        test_agg = test_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)
        final_b_acc = balanced_accuracy_score(test_agg["true_label"], test_preds)
        acc = accuracy_score(test_agg["true_label"], test_preds)
        f1 = f1_score(test_agg["true_label"], test_preds)
        recall_0 = recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0)
        recall_1 = recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0)
        precision = precision_score(test_agg["true_label"], test_preds, zero_division=0)

        results_list.append({
            "Model": model_name,
            "Strategy": strategy_name,
            "Thresh": round(best_thresh, 2),
            "Bal_Acc": round(final_b_acc, 4),
            "Accuracy": round(acc, 2),
            "F1": round(f1, 2),
            "Rec_Relaxed": round(recall_0, 2),
            "Rec_Stress": round(recall_1, 2),
            "Precision": round(precision, 2)
        })

comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by="Bal_Acc", ascending=False).reset_index(drop=True)

print("\n=== Wav2Vec 2.0 Results ===")
print(comparison_df.to_string(index=False))



=== Wav2Vec 2.0 Results ===
                 Model        Strategy  Thresh  Bal_Acc  Accuracy   F1  Rec_Relaxed  Rec_Stress  Precision
               XGBoost            Mean    0.76   0.5591      0.58 0.68         0.50        0.62       0.76
   Logistic Regression             Max    0.74   0.5455      0.74 0.85         0.09        1.00       0.73
   Logistic Regression          Median    0.70   0.5273      0.52 0.60         0.55        0.51       0.74
   Logistic Regression 75th_Percentile    0.86   0.5182      0.51 0.59         0.55        0.49       0.73
Balanced Random Forest          Median    0.50   0.5091      0.53 0.63         0.45        0.56       0.72
         Random Forest            Mean    0.70   0.5045      0.55 0.65         0.41        0.60       0.72
         Random Forest          Median    0.70   0.5000      0.56 0.67         0.36        0.64       0.71
Balanced Random Forest            Mean    0.50   0.4818      0.49 0.59         0.45        0.51       0.70
        

In [3]:
evaluate_embeddings("wav2vec_embeddings_original.csv", feature_regex='^w2v_')


=== Results: wav2vec_embeddings_original (5-Fold CV, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
Balanced Random Forest 75th_Percentile   0.52 0.5839 ± 0.0861 0.6418 ± 0.1243 0.7295 ± 0.1154 0.4454 ± 0.0876 0.7223 ± 0.1860 0.7539 ± 0.0449
               XGBoost          Median   0.80 0.5600 ± 0.1028 0.5958 ± 0.0912 0.6885 ± 0.0833 0.4743 ± 0.2098 0.6457 ± 0.1338 0.7543 ± 0.0717
Balanced Random Forest            Mean   0.46 0.5520 ± 0.0897 0.5979 ± 0.1322 0.6835 ± 0.1383 0.4413 ± 0.1345 0.6628 ± 0.2092 0.7351 ± 0.0538
               XGBoost            Mean   0.73 0.5493 ± 0.1185 0.5855 ± 0.1350 0.6751 ± 0.1317 0.4610 ± 0.1227 0.6375 ± 0.1724 0.7325 ± 0.0839
         Random Forest 75th_Percentile   0.78 0.5458 ± 0.0878 0.5343 ± 0.1302 0.5899 ± 0.1697 0.5674 ± 0.1956 0.5242 ± 0.2245 0.7328 ± 0.0973
         Random Forest          Median   0.70 0.5375 ± 0.0801 0.5919 ± 0.1176 

In [4]:
evaluate_embeddings("wav2vec_embeddings_original.csv", feature_regex='^w2v_', pca_components=0.95)


=== Results: wav2vec_embeddings_original (5-Fold CV, PCA=0.95, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
               XGBoost          Median   0.74 0.5544 ± 0.0585 0.5743 ± 0.0867 0.6495 ± 0.1052 0.5093 ± 0.2677 0.5995 ± 0.2090 0.7585 ± 0.0474
Balanced Random Forest 75th_Percentile   0.49 0.5397 ± 0.0644 0.5901 ± 0.1140 0.6761 ± 0.1403 0.4174 ± 0.2183 0.6621 ± 0.2232 0.7304 ± 0.0455
   Logistic Regression          Median   0.58 0.5351 ± 0.0506 0.4943 ± 0.0790 0.5247 ± 0.1305 0.6307 ± 0.2853 0.4396 ± 0.2199 0.7755 ± 0.0874
   Logistic Regression            Mean   0.53 0.5329 ± 0.0765 0.5294 ± 0.0934 0.5906 ± 0.1319 0.5396 ± 0.2762 0.5262 ± 0.2157 0.7536 ± 0.1039
Balanced Random Forest            Mean   0.47 0.5297 ± 0.0753 0.5564 ± 0.0958 0.6371 ± 0.1125 0.4643 ± 0.2688 0.5952 ± 0.2176 0.7402 ± 0.0597
Balanced Random Forest          Median   0.50 0.5264 ± 0.0678 0.5042

In [7]:
param_grid = {
    "clf__n_estimators":      [100, 200, 500],
    "clf__max_depth":         [None, 10, 20],
    "clf__min_samples_leaf":  [1, 2, 4],
    "clf__max_features":      ["sqrt", "log2"],
}

results = evaluate_best_model_grid_search(
    filepath="hubert_embeddings_original.csv",
    feature_regex='^hubert_',
    model_name="Balanced Random Forest",
    aggregation_strategy="75th_Percentile",
    param_grid=param_grid,
)

Fold 0 best params: {'clf__max_depth': 20, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 4, 'clf__n_estimators': 200}
Fold 1 best params: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__n_estimators': 200}
Fold 2 best params: {'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 4, 'clf__n_estimators': 500}
Fold 3 best params: {'clf__max_depth': 20, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 2, 'clf__n_estimators': 500}
Fold 4 best params: {'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 1, 'clf__n_estimators': 500}

=== Balanced Random Forest + GridSearch (75th_Percentile, 5-Fold CV) ===
 Fold                                                                                                 Best_Params  Thresh  Bal_Acc  Accuracy       F1  Rec_Relaxed  Rec_Stress  Precision
    0   {'clf__max_depth': 20, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 4, 'clf__n_estimato

## HuBert

In [3]:
from transformers import Wav2Vec2FeatureExtractor, HubertModel
from tqdm import tqdm

AUDIO_WINDOWS_DIR = "./data/5s_0overlap_original" 
GROUND_TRUTH_CSV = "../labels_v2.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "facebook/hubert-base-ls960"
# HuBERT uses the same feature extractor logic as Wav2Vec2 for raw audio
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = HubertModel.from_pretrained(model_name).to(device)
model.eval()

target_sample_rate = 16000
embedding_list = []

print("Extracting HuBERT embeddings...")

with torch.no_grad():
    for subject in tqdm(sorted(os.listdir(AUDIO_WINDOWS_DIR)), desc="Subjects"):
        subject_dir = os.path.join(AUDIO_WINDOWS_DIR, subject)
        if not os.path.isdir(subject_dir): continue
            
        for subject_activity in sorted(os.listdir(subject_dir)):
            activity_dir = os.path.join(subject_dir, subject_activity)
            if not os.path.isdir(activity_dir): continue
                
            # Using the fix we established earlier
            activity = subject_activity.split('_')[1]
            
            for window_file in sorted(os.listdir(activity_dir)):
                if not window_file.lower().endswith(".wav"): continue
                    
                window_id = int(window_file.split("_")[1].split(".")[0])
                file_path = os.path.join(activity_dir, window_file)
                
                # Load audio
                waveform, sample_rate = torchaudio.load(file_path)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                    
                if sample_rate != target_sample_rate:
                    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
                    waveform = resampler(waveform)
                    
                # Process and Forward Pass
                inputs = processor(waveform.squeeze().numpy(), sampling_rate=target_sample_rate, return_tensors="pt")
                inputs = {key: val.to(device) for key, val in inputs.items()}
                outputs = model(**inputs)
                
                # Temporal Pooling
                mean_pooled_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                
                # Store results
                row_data = {
                    "subject": subject,
                    "activity": activity,
                    "subject_activity": subject_activity,
                    "window_id": window_id
                }
                
                # Use 'hubert_' prefix just to keep things clean
                for i, val in enumerate(mean_pooled_embedding):
                    row_data[f"hubert_{i}"] = val
                    
                embedding_list.append(row_data)

embeddings_df = pd.DataFrame(embedding_list)

print("Merging with ground truth labels...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";")

merged_df = pd.merge(
    embeddings_df, 
    gt_df, 
    left_on="subject_activity", 
    right_on="subject/task", 
    how="inner"
)

output_csv = "hubert_embeddings_original.csv"
merged_df.to_csv(output_csv, index=False)
print(f"Extraction complete! Saved {len(merged_df)} windows to {output_csv}.")

Using device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Extracting HuBERT embeddings...


Subjects: 100%|██████████| 54/54 [02:15<00:00,  2.50s/it]


Merging with ground truth labels...
Extraction complete! Saved 4536 windows to hubert_embeddings_original.csv.


In [3]:
import warnings

# Suppress minor warnings for clean output
warnings.filterwarnings("ignore")

df = pd.read_csv("hubert_embeddings_original.csv")

# Extract features, target, subject groups for splitting, and task IDs for aggregation
X = df.filter(regex='^hubert_')
y = df["binary-stress"]
groups = df["subject"]
task_ids = df["subject_activity"]

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups))

train_subjects = set(groups.iloc[train_idx])
test_subjects = set(groups.iloc[test_idx])
overlap = train_subjects & test_subjects
assert not overlap, f"Subject leakage detected: {sorted(overlap)}"

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
task_ids_train, task_ids_test = task_ids.iloc[train_idx], task_ids.iloc[test_idx]

# Calculate imbalance weight for XGBoost
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Define Models
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Balanced Random Forest": make_pipeline(
        BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        StandardScaler(),
        XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

agg_strategies = {
    "Mean": "mean",
    "Median": "median",
    "Max": "max",
    "75th_Percentile": lambda x: np.percentile(x, 75)
}

cv_train = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
results_list = []


for model_name, model in models.items():

    # Train and predict
    model.fit(X_train, y_train)
    test_probs = model.predict_proba(X_test)[:, 1]

    # Out-of-fold probabilities for unbiased threshold tuning, grouped by subject
    train_unbiased_probs = cross_val_predict(
        model, X_train, y_train,
        groups=groups_train,
        cv=cv_train,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    train_base_df = pd.DataFrame({
        "subject_activity": task_ids_train.values,
        "true_label": y_train.values,
        "pred_prob": train_unbiased_probs
    })

    test_base_df = pd.DataFrame({
        "subject_activity": task_ids_test.values,
        "true_label": y_test.values,
        "pred_prob": test_probs
    })

    # Test Aggregation Strategies
    for strategy_name, agg_func in agg_strategies.items():

        train_agg = train_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        best_thresh = 0.5
        best_b_acc = 0
        for thresh in np.arange(0.1, 0.9, 0.02):
            train_preds = (train_agg["agg_prob"] >= thresh).astype(int)
            b_acc = balanced_accuracy_score(train_agg["true_label"], train_preds)
            if b_acc >= best_b_acc:
                best_b_acc = b_acc
                best_thresh = thresh

        test_agg = test_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)
        final_b_acc = balanced_accuracy_score(test_agg["true_label"], test_preds)
        acc = accuracy_score(test_agg["true_label"], test_preds)
        f1 = f1_score(test_agg["true_label"], test_preds)
        recall_0 = recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0)
        recall_1 = recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0)
        precision = precision_score(test_agg["true_label"], test_preds, zero_division=0)

        results_list.append({
            "Model": model_name,
            "Strategy": strategy_name,
            "Thresh": round(best_thresh, 2),
            "Bal_Acc": round(final_b_acc, 4),
            "Accuracy": round(acc, 2),
            "F1": round(f1, 2),
            "Rec_Relaxed": round(recall_0, 2),
            "Rec_Stress": round(recall_1, 2),
            "Precision": round(precision, 2)
        })

comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by="Bal_Acc", ascending=False).reset_index(drop=True)

print("\n=== Hubert Results ===")
print(comparison_df.to_string(index=False))



=== Hubert Results ===
                 Model        Strategy  Thresh  Bal_Acc  Accuracy   F1  Rec_Relaxed  Rec_Stress  Precision
Balanced Random Forest            Mean    0.44   0.6409      0.74 0.83         0.41        0.87       0.79
Balanced Random Forest          Median    0.44   0.6227      0.71 0.81         0.41        0.84       0.78
               XGBoost 75th_Percentile    0.88   0.6227      0.75 0.84         0.32        0.93       0.77
Balanced Random Forest 75th_Percentile    0.46   0.6000      0.74 0.84         0.27        0.93       0.76
         Random Forest             Max    0.80   0.5864      0.70 0.80         0.32        0.85       0.76
   Logistic Regression            Mean    0.62   0.5545      0.52 0.58         0.64        0.47       0.76
         Random Forest          Median    0.64   0.5500      0.73 0.83         0.14        0.96       0.74
   Logistic Regression             Max    0.88   0.5318      0.70 0.82         0.14        0.93       0.73
         Rand

In [3]:
evaluate_embeddings("hubert_embeddings_original.csv", feature_regex='^hubert_')


=== Results: hubert_embeddings_original (5-Fold CV, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
               XGBoost            Mean   0.72 0.5432 ± 0.0601 0.5873 ± 0.1027 0.6752 ± 0.1235 0.4387 ± 0.1276 0.6477 ± 0.1845 0.7332 ± 0.0305
         Random Forest          Median   0.73 0.5342 ± 0.0772 0.5306 ± 0.1061 0.5860 ± 0.1493 0.5502 ± 0.2286 0.5182 ± 0.2091 0.7494 ± 0.0530
Balanced Random Forest            Mean   0.49 0.5256 ± 0.0589 0.5179 ± 0.1047 0.5689 ± 0.1559 0.5481 ± 0.2355 0.5031 ± 0.2230 0.7470 ± 0.0665
Balanced Random Forest          Median   0.48 0.5218 ± 0.0438 0.5239 ± 0.0773 0.5867 ± 0.1424 0.5204 ± 0.2219 0.5233 ± 0.1884 0.7452 ± 0.0573
Balanced Random Forest 75th_Percentile   0.53 0.5218 ± 0.0766 0.5299 ± 0.1037 0.6011 ± 0.1249 0.5047 ± 0.2262 0.5389 ± 0.1998 0.7317 ± 0.0721
         Random Forest            Mean   0.70 0.5163 ± 0.0710 0.5597 ± 0.1047 0

In [4]:
evaluate_embeddings("hubert_embeddings_original.csv", feature_regex='^hubert_', pca_components=0.95)


=== Results: hubert_embeddings_original (5-Fold CV, PCA=0.95, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
               XGBoost          Median   0.77 0.5288 ± 0.0780 0.5738 ± 0.1134 0.6588 ± 0.1350 0.4257 ± 0.2212 0.6319 ± 0.2163 0.7261 ± 0.0536
         Random Forest            Mean   0.68 0.5143 ± 0.0338 0.6182 ± 0.0584 0.7338 ± 0.0598 0.2653 ± 0.1505 0.7632 ± 0.1371 0.7174 ± 0.0157
Balanced Random Forest             Max   0.58 0.5046 ± 0.0435 0.5010 ± 0.1522 0.5237 ± 0.2323 0.5113 ± 0.3061 0.4979 ± 0.3334 0.7131 ± 0.0702
Balanced Random Forest            Mean   0.43 0.4977 ± 0.0557 0.6104 ± 0.0809 0.7305 ± 0.0738 0.2277 ± 0.1162 0.7677 ± 0.1435 0.7060 ± 0.0250
Balanced Random Forest          Median   0.46 0.4921 ± 0.0618 0.5332 ± 0.1091 0.6018 ± 0.1936 0.4024 ± 0.2792 0.5819 ± 0.2504 0.7086 ± 0.0311
Balanced Random Forest 75th_Percentile   0.52 0.4895 ± 0.0534 0.4673 

## AST

Vision Transformer applied on audio, which is turned into a spectrogram (an image).

In [5]:
from transformers import ASTFeatureExtractor, ASTModel
from tqdm import tqdm

AUDIO_WINDOWS_DIR = "./data/5s_0overlap_original" 
GROUND_TRUTH_CSV = "../labels_v2.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We use the standard AST model fine-tuned on AudioSet
# login()  # Optional: uncomment if Hugging Face authentication is needed

model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
processor = ASTFeatureExtractor.from_pretrained(model_name)
model = ASTModel.from_pretrained(model_name).to(device)
model.eval() # Freeze the model

target_sample_rate = 16000
embedding_list = []

print("Extracting AST embeddings...")

# 3. Extraction Loop
with torch.no_grad():
    for subject in tqdm(sorted(os.listdir(AUDIO_WINDOWS_DIR)), desc="Subjects"):
        subject_dir = os.path.join(AUDIO_WINDOWS_DIR, subject)
        if not os.path.isdir(subject_dir): continue
            
        for subject_activity in sorted(os.listdir(subject_dir)):
            activity_dir = os.path.join(subject_dir, subject_activity)
            if not os.path.isdir(activity_dir): continue
                
            # Using the established fix for the ID
            activity = subject_activity.split("_")[1]
            
            for window_file in sorted(os.listdir(activity_dir)):
                if not window_file.lower().endswith(".wav"): continue
                    
                window_id = int(window_file.split("_")[1].split(".")[0])
                file_path = os.path.join(activity_dir, window_file)
                
                # Load audio
                waveform, sample_rate = torchaudio.load(file_path)
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)
                    
                if sample_rate != target_sample_rate:
                    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
                    waveform = resampler(waveform)
                    
                # Process and Forward Pass
                # AST expects the raw waveform and handles the spectrogram conversion internally
                inputs = processor(waveform.squeeze().numpy(), sampling_rate=target_sample_rate, return_tensors="pt")
                inputs = {key: val.to(device) for key, val in inputs.items()}
                outputs = model(**inputs)
                
                # Temporal/Patch Pooling
                # AST outputs sequence of patches. We take the mean across the sequence.
                mean_pooled_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                
                # Store results
                row_data = {
                    "subject": subject,
                    "activity": activity,
                    "subject_activity": subject_activity,
                    "window_id": window_id
                }
                
                # Prefix with 'ast_'
                for i, val in enumerate(mean_pooled_embedding):
                    row_data[f"ast_{i}"] = val
                    
                embedding_list.append(row_data)

# 4. Create DataFrame and Merge with Ground Truth
embeddings_df = pd.DataFrame(embedding_list)

print("Merging with ground truth labels...")
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";")

merged_df = pd.merge(
    embeddings_df, 
    gt_df, 
    left_on="subject_activity", 
    right_on="subject/task", 
    how="inner"
)

output_csv = "ast_embeddings_original.csv"
merged_df.to_csv(output_csv, index=False)
print(f"Extraction complete! Saved {len(merged_df)} windows to {output_csv}.")

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting AST embeddings...


Subjects: 100%|██████████| 54/54 [07:04<00:00,  7.87s/it]


Merging with ground truth labels...
Extraction complete! Saved 4536 windows to ast_embeddings_original.csv.


In [ ]:
import warnings

# Suppress minor warnings for clean output
warnings.filterwarnings("ignore")

df = pd.read_csv("ast_embeddings_original.csv")

# Extract features, target, subject groups for splitting, and task IDs for aggregation
X = df.filter(regex='^ast_')
y = df["binary-stress"]
groups = df["subject"]
task_ids = df["subject_activity"]

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups))

train_subjects = set(groups.iloc[train_idx])
test_subjects = set(groups.iloc[test_idx])
overlap = train_subjects & test_subjects
assert not overlap, f"Subject leakage detected: {sorted(overlap)}"

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
task_ids_train, task_ids_test = task_ids.iloc[train_idx], task_ids.iloc[test_idx]

# Calculate imbalance weight for XGBoost
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Define Models
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Balanced Random Forest": make_pipeline(
        BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='auto', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        StandardScaler(),
        XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

agg_strategies = {
    "Mean": "mean",
    "Median": "median",
    "Max": "max",
    "75th_Percentile": lambda x: np.percentile(x, 75)
}

cv_train = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
results_list = []


for model_name, model in models.items():

    # Train and predict
    model.fit(X_train, y_train)
    test_probs = model.predict_proba(X_test)[:, 1]

    # Out-of-fold probabilities for unbiased threshold tuning, grouped by subject
    train_unbiased_probs = cross_val_predict(
        model, X_train, y_train,
        groups=groups_train,
        cv=cv_train,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    train_base_df = pd.DataFrame({
        "subject_activity": task_ids_train.values,
        "true_label": y_train.values,
        "pred_prob": train_unbiased_probs
    })

    test_base_df = pd.DataFrame({
        "subject_activity": task_ids_test.values,
        "true_label": y_test.values,
        "pred_prob": test_probs
    })

    # Test Aggregation Strategies
    for strategy_name, agg_func in agg_strategies.items():

        train_agg = train_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        best_thresh = 0.5
        best_b_acc = 0
        for thresh in np.arange(0.1, 0.9, 0.02):
            train_preds = (train_agg["agg_prob"] >= thresh).astype(int)
            b_acc = balanced_accuracy_score(train_agg["true_label"], train_preds)
            if b_acc >= best_b_acc:
                best_b_acc = b_acc
                best_thresh = thresh

        test_agg = test_base_df.groupby("subject_activity").agg(
            true_label=("true_label", "first"),
            agg_prob=("pred_prob", agg_func)
        ).reset_index()

        test_preds = (test_agg["agg_prob"] >= best_thresh).astype(int)
        final_b_acc = balanced_accuracy_score(test_agg["true_label"], test_preds)
        acc = accuracy_score(test_agg["true_label"], test_preds)
        f1 = f1_score(test_agg["true_label"], test_preds)
        recall_0 = recall_score(test_agg["true_label"], test_preds, pos_label=0, zero_division=0)
        recall_1 = recall_score(test_agg["true_label"], test_preds, pos_label=1, zero_division=0)
        precision = precision_score(test_agg["true_label"], test_preds, zero_division=0)

        results_list.append({
            "Model": model_name,
            "Strategy": strategy_name,
            "Thresh": round(best_thresh, 2),
            "Bal_Acc": round(final_b_acc, 4),
            "Accuracy": round(acc, 2),
            "F1": round(f1, 2),
            "Rec_Relaxed": round(recall_0, 2),
            "Rec_Stress": round(recall_1, 2),
            "Precision": round(precision, 2)
        })

comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by="Bal_Acc", ascending=False).reset_index(drop=True)

print("\n=== AST Results ===")
print(comparison_df.to_string(index=False))



=== AST Results ===
                 Model        Strategy  Thresh  Bal_Acc  Accuracy   F1  Rec_Relaxed  Rec_Stress  Precision
Balanced Random Forest          Median    0.48   0.5227      0.57 0.68         0.41        0.64       0.73
Balanced Random Forest 75th_Percentile    0.48   0.5045      0.62 0.75         0.23        0.78       0.72
         Random Forest 75th_Percentile    0.56   0.5000      0.71 0.83         0.00        1.00       0.71
               XGBoost             Max    0.54   0.5000      0.71 0.83         0.00        1.00       0.71
         Random Forest          Median    0.50   0.4909      0.70 0.82         0.00        0.98       0.71
         Random Forest            Mean    0.50   0.4909      0.70 0.82         0.00        0.98       0.71
Balanced Random Forest            Mean    0.28   0.4909      0.70 0.82         0.00        0.98       0.71
   Logistic Regression            Mean    0.40   0.4818      0.61 0.74         0.18        0.78       0.70
               X

: 

In [4]:
evaluate_embeddings("ast_embeddings_original.csv", feature_regex='^ast_')


=== Results: ast_embeddings_original (5-Fold CV, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
         Random Forest             Max   0.83 0.5480 ± 0.0511 0.5455 ± 0.1099 0.5885 ± 0.1900 0.5573 ± 0.2697 0.5386 ± 0.2521 0.7906 ± 0.1101
Balanced Random Forest             Max   0.69 0.5289 ± 0.0490 0.4714 ± 0.1394 0.4366 ± 0.2797 0.6625 ± 0.3012 0.3953 ± 0.3123 0.8322 ± 0.1403
Balanced Random Forest          Median   0.60 0.5217 ± 0.0675 0.3891 ± 0.0449 0.2781 ± 0.1653 0.8484 ± 0.2819 0.1950 ± 0.1614 0.9136 ± 0.1351
   Logistic Regression          Median   0.70 0.5128 ± 0.0536 0.5016 ± 0.0801 0.5575 ± 0.1274 0.5391 ± 0.2771 0.4864 ± 0.2083 0.7333 ± 0.0547
   Logistic Regression 75th_Percentile   0.69 0.5105 ± 0.0515 0.6122 ± 0.0703 0.7270 ± 0.0683 0.2708 ± 0.1580 0.7501 ± 0.1389 0.7152 ± 0.0254
         Random Forest 75th_Percentile   0.81 0.5092 ± 0.0401 0.4475 ± 0.0885 0.42

In [5]:
evaluate_embeddings("ast_embeddings_original.csv", feature_regex='^ast_', pca_components=0.95)


=== Results: ast_embeddings_original (5-Fold CV, PCA=0.95, mean ± std) ===
                 Model        Strategy Thresh         Bal_Acc        Accuracy              F1     Rec_Relaxed      Rec_Stress       Precision
   Logistic Regression 75th_Percentile   0.73 0.5371 ± 0.0845 0.5195 ± 0.0992 0.5534 ± 0.1622 0.5888 ± 0.3488 0.4853 ± 0.2517 0.7987 ± 0.1225
   Logistic Regression             Max   0.75 0.5261 ± 0.0564 0.6055 ± 0.0772 0.7042 ± 0.0954 0.3383 ± 0.2856 0.7138 ± 0.2137 0.7349 ± 0.0395
         Random Forest             Max   0.80 0.5253 ± 0.0378 0.5369 ± 0.1339 0.5725 ± 0.2297 0.4981 ± 0.3067 0.5525 ± 0.3093 0.7726 ± 0.1148
         Random Forest            Mean   0.78 0.5181 ± 0.0632 0.3795 ± 0.1146 0.2437 ± 0.2509 0.8526 ± 0.1507 0.1837 ± 0.2073 0.6017 ± 0.3431
Balanced Random Forest            Mean   0.52 0.5101 ± 0.0794 0.3774 ± 0.0580 0.2868 ± 0.1241 0.8332 ± 0.1919 0.1871 ± 0.0907 0.8085 ± 0.1644
         Random Forest 75th_Percentile   0.77 0.5080 ± 0.0564 0.4919 ± 0

## Subject-independent CV evaluation

In [ ]:

# ============================================================
# Subject-independent 5-fold outer CV, task-level metrics
#
# Nested inner-CV grid search per model. The inner-CV scoring
# objective is balanced accuracy AT THE TASK LEVEL: inner-OOF
# window probabilities are aggregated to the task level using
# INNER_SCORING_AGG (mean) before scoring, so hyperparameter
# selection matches the unit reported on the outer fold and is
# decoupled from the choice of final aggregation. The outer fold
# then evaluates the tuned model under all four aggregation
# strategies (mean / median / max / p75).
# ============================================================

import os
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, GridSearchCV, ParameterGrid
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

OUTER_SPLITS = 5
INNER_SPLITS = 3
PCA_N_COMPONENTS = 0.95
THRESHOLDS = np.arange(0.1, 0.9, 0.02)

# Aggregation used INSIDE the inner-CV grid search to score candidates
# at the task level. The outer fold separately evaluates all four
# aggregations. Mean is the default because it is picklable and stable;
# the same default is used in the aligned visual notebook.
INNER_SCORING_AGG = "mean"
# Threshold used inside the inner CV when ranking candidates. Kept at
# 0.5 so hyperparameter selection is decoupled from threshold tuning
# (which still happens on outer-training OOF probabilities afterwards).
INNER_SCORING_THRESHOLD = 0.5


def _safe_grouped_cv(y, groups, max_splits=5):
    """StratifiedGroupKFold with a safe number of folds for small grouped datasets."""
    y = pd.Series(y).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)

    n_splits = min(max_splits, int(y.value_counts().min()), int(groups.nunique()))
    if n_splits < 2:
        raise ValueError("Not enough class examples/groups for grouped cross-validation.")

    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


def _load_binary_labels(gt_path="../labels_v2.csv"):
    gt = pd.read_csv(gt_path, sep=";")
    gt = gt.rename(columns={"subject/task": "subject_activity", "binary-stress": "y"})
    gt = gt[["subject_activity", "y"]].dropna().copy()
    gt["subject_activity"] = gt["subject_activity"].astype(str)
    gt["subject"] = gt["subject_activity"].apply(lambda x: str(x).split("_", 1)[0])
    gt["y"] = gt["y"].astype(int)
    return gt[["subject", "subject_activity", "y"]]


def _normalize_audio_keys(df):
    df = df.copy()
    if "subject_activity" not in df.columns:
        if {"subject", "activity"}.issubset(df.columns):
            df["subject"] = df["subject"].astype(str)
            df["activity"] = df["activity"].astype(str)
            df["subject_activity"] = df["subject"] + "_" + df["activity"]
        elif "subject/task" in df.columns:
            df = df.rename(columns={"subject/task": "subject_activity"})
        else:
            raise ValueError("Expected subject_activity, subject/task, or subject + activity columns.")

    df["subject_activity"] = df["subject_activity"].astype(str)
    if "subject" not in df.columns:
        df["subject"] = df["subject_activity"].apply(lambda x: str(x).split("_", 1)[0])
    df["subject"] = df["subject"].astype(str)

    if "window_id" not in df.columns:
        raise ValueError("Expected a window_id column for window-to-task aggregation.")

    return df


def load_audio_window_table(csv_path, gt_path="../labels_v2.csv"):
    """Load an audio window-feature CSV and attach binary labels if needed."""
    df = pd.read_csv(csv_path)
    df = _normalize_audio_keys(df)

    if "y" not in df.columns:
        if "binary-stress" in df.columns:
            df = df.rename(columns={"binary-stress": "y"})
        elif "target" in df.columns:
            df = df.rename(columns={"target": "y"})
        else:
            gt = _load_binary_labels(gt_path)
            df = df.merge(gt, on=["subject", "subject_activity"], how="inner")

    df["y"] = df["y"].astype(int)
    return df


def infer_audio_feature_cols(df, feature_prefix=None):
    protected = {
        "subject", "activity", "subject_activity", "subject/task", "window_id",
        "y", "target", "binary-stress", "affect3-class", "affect3-class-v2",
    }
    if feature_prefix is not None:
        cols = [c for c in df.columns if str(c).startswith(feature_prefix)]
    else:
        cols = [c for c in df.columns if c not in protected]

    # Keep numeric columns only.
    out = []
    for c in cols:
        converted = pd.to_numeric(df[c], errors="coerce")
        if converted.notna().any():
            df[c] = converted
            out.append(c)
    return out


def make_audio_preprocessor(use_pca=False):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
    if use_pca:
        steps.append(("pca", PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)))

    return Pipeline(steps)


def make_audio_model_grids(y_train, use_pca=False):
    """Candidate audio pipelines and small hyperparameter grids tuned inside each outer fold."""
    scale_weight = (pd.Series(y_train).eq(0).sum()) / max(pd.Series(y_train).eq(1).sum(), 1)
    pre = make_audio_preprocessor(use_pca=use_pca)

    return OrderedDict({
        "logreg": (
            make_pipeline(
                clone(pre),
                LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
            ),
            {
                "logisticregression__C": [0.01, 0.1, 1.0, 10.0],
            },
        ),
        "balanced_rf": (
            make_pipeline(
                clone(pre),
                BalancedRandomForestClassifier(
                    sampling_strategy="auto",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
            {
                "balancedrandomforestclassifier__n_estimators": [200, 500],
                "balancedrandomforestclassifier__max_depth": [None, 5, 10],
                "balancedrandomforestclassifier__min_samples_leaf": [1, 3],
            },
        ),
        "random_forest": (
            make_pipeline(
                clone(pre),
                RandomForestClassifier(
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
            {
                "randomforestclassifier__n_estimators": [200, 500],
                "randomforestclassifier__max_depth": [None, 5, 10],
                "randomforestclassifier__min_samples_leaf": [1, 3],
            },
        ),
        "xgboost": (
            make_pipeline(
                clone(pre),
                XGBClassifier(
                    scale_pos_weight=scale_weight,
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
            {
                "xgbclassifier__n_estimators": [100, 300],
                "xgbclassifier__max_depth": [2, 3],
                "xgbclassifier__learning_rate": [0.03, 0.1],
                "xgbclassifier__subsample": [0.8, 1.0],
            },
        ),
        "mlp": (
            make_pipeline(
                clone(pre),
                MLPClassifier(
                    activation="relu",
                    learning_rate_init=1e-3,
                    max_iter=1000,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                ),
            ),
            {
                "mlpclassifier__hidden_layer_sizes": [(32,), (32, 8)],
                "mlpclassifier__alpha": [1e-4, 1e-3],
            },
        ),
    })


def _task_level_inner_score(model, X, y, groups, task_ids, inner_cv, agg_func, threshold):
    """Inner-CV task-level balanced accuracy for one model configuration.

    Aggregates inner-OOF window probabilities to the task level using
    ``agg_func``, thresholds at ``threshold``, and returns balanced accuracy
    over tasks. Returns NaN if cross_val_predict raises.
    """
    try:
        oof = cross_val_predict(
            clone(model), X, y, groups=groups, cv=inner_cv,
            method="predict_proba", n_jobs=-1,
        )[:, 1]
    except Exception:
        return np.nan

    task_df = pd.DataFrame({
        "task": pd.Series(task_ids).values,
        "true_label": pd.Series(y).values,
        "prob": oof,
    })
    agg = task_df.groupby("task").agg(
        true_label=("true_label", "first"),
        agg_prob=("prob", agg_func),
    )
    preds = (agg["agg_prob"] >= threshold).astype(int)
    return balanced_accuracy_score(agg["true_label"], preds)


def tune_audio_model_inner_cv(
    model, param_grid, X_train, y_train, groups_train, task_ids_train,
    inner_cv, agg_func, scoring_threshold=INNER_SCORING_THRESHOLD,
):
    """Per-model inner-CV grid search with task-level balanced accuracy objective.

    For each candidate parameter combination, inner-OOF window
    probabilities are aggregated per task using ``agg_func`` and scored
    with balanced accuracy at the task level. Returns the refitted best
    estimator, its parameters, and the best inner-CV score. Mirrors
    ``grid_search_window_task_level`` in the visual notebook.
    """
    best_score = -np.inf
    best_params = None

    for params in ParameterGrid(param_grid):
        candidate = clone(model).set_params(**params)
        score = _task_level_inner_score(
            candidate, X_train, y_train, groups_train, task_ids_train,
            inner_cv=inner_cv, agg_func=agg_func, threshold=scoring_threshold,
        )
        if not np.isnan(score) and score > best_score:
            best_score = score
            best_params = dict(params)

    if best_params is None:
        best_params = {}
        best_score = np.nan

    tuned = clone(model).set_params(**best_params)
    tuned.fit(X_train, y_train)
    return tuned, best_params, float(best_score) if not np.isnan(best_score) else np.nan


def tune_threshold(y_true, probs, thresholds=THRESHOLDS):
    best_thresh = 0.5
    best_bacc = -np.inf
    for thresh in thresholds:
        preds = (np.asarray(probs) >= thresh).astype(int)
        bacc = balanced_accuracy_score(y_true, preds)
        if bacc >= best_bacc:
            best_bacc = bacc
            best_thresh = float(thresh)
    return best_thresh


def compute_metrics(y_true, probs, threshold):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    probs = np.asarray(probs)
    preds = (probs >= threshold).astype(int)
    row = {
        "accuracy": accuracy_score(y_true, preds),
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds, zero_division=0),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall_no_stress": recall_score(y_true, preds, pos_label=0, zero_division=0),
        "recall_stress": recall_score(y_true, preds, pos_label=1, zero_division=0),
    }
    row["roc_auc"] = roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan
    return row


def summarize_cv_results(fold_df, group_cols):
    metric_cols = [
        "accuracy", "balanced_accuracy", "f1", "precision",
        "recall_no_stress", "recall_stress", "roc_auc",
    ]
    rows = []
    for keys, group in fold_df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for metric in metric_cols:
            row[f"{metric}_mean"] = group[metric].mean()
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        row["n_folds"] = group["fold"].nunique()
        row["n_tasks_mean"] = group["n_test_tasks"].mean()
        rows.append(row)
    return (
        pd.DataFrame(rows)
        .sort_values(["balanced_accuracy_mean", "f1_mean", "accuracy_mean"], ascending=False)
        .reset_index(drop=True)
    )


def evaluate_audio_window_to_task_cv(
    df,
    feature_cols,
    feature_set,
    use_pca=False,
    outer_splits=OUTER_SPLITS,
    inner_splits=INNER_SPLITS,
):
    """Train on audio windows, aggregate probabilities to task level, and score held-out subjects."""
    df = df.copy()
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    task_units = df[["subject", "subject_activity", "y"]].drop_duplicates().reset_index(drop=True)
    outer_cv = _safe_grouped_cv(task_units["y"], task_units["subject"], max_splits=outer_splits)

    aggregation_strategies = OrderedDict({
        "mean": "mean",
        "median": "median",
        "max": "max",
        "p75": lambda x: np.percentile(x, 75),
    })

    # Aggregation function used inside the inner-CV grid search to score
    # candidates at the task level. Pulled from the same dict so the
    # implementation is consistent with the outer aggregations.
    if INNER_SCORING_AGG not in aggregation_strategies:
        raise ValueError(
            f"INNER_SCORING_AGG={INNER_SCORING_AGG!r} not in aggregation_strategies."
        )
    inner_agg_func = aggregation_strategies[INNER_SCORING_AGG]

    fold_rows = []
    prediction_rows = []
    for fold, (task_train_idx, task_test_idx) in enumerate(
        outer_cv.split(task_units, task_units["y"], task_units["subject"]), start=1
    ):
        train_subjects = set(task_units.loc[task_train_idx, "subject"])
        test_subjects = set(task_units.loc[task_test_idx, "subject"])
        assert train_subjects.isdisjoint(test_subjects), "Subject leakage detected."

        train_df = df[df["subject"].isin(train_subjects)].reset_index(drop=True)
        test_df = df[df["subject"].isin(test_subjects)].reset_index(drop=True)

        X_train = train_df[feature_cols]
        y_train = train_df["y"].astype(int)
        g_train = train_df["subject"]
        task_train = train_df["subject_activity"]
        X_test = test_df[feature_cols]
        y_test = test_df["y"].astype(int)

        inner_cv = _safe_grouped_cv(y_train, g_train, max_splits=inner_splits)
        model_grids = make_audio_model_grids(y_train, use_pca=use_pca)

        for model_name, (base_model, param_grid) in model_grids.items():
            tuned_model, best_params, inner_best_score = tune_audio_model_inner_cv(
                base_model,
                param_grid,
                X_train,
                y_train,
                g_train,
                task_train,
                inner_cv,
                inner_agg_func,
                scoring_threshold=INNER_SCORING_THRESHOLD,
            )

            # Inner-OOF window probabilities of the selected configuration,
            # used only to tune the per-aggregation decision threshold below.
            train_oof_probs = cross_val_predict(
                clone(tuned_model),
                X_train,
                y_train,
                groups=g_train,
                cv=inner_cv,
                method="predict_proba",
                n_jobs=-1,
            )[:, 1]

            tuned_model.fit(X_train, y_train)
            test_probs = tuned_model.predict_proba(X_test)[:, 1]

            train_base = pd.DataFrame({
                "subject_activity": train_df["subject_activity"].values,
                "true_label": y_train.values,
                "pred_prob": train_oof_probs,
            })
            test_base = pd.DataFrame({
                "subject_activity": test_df["subject_activity"].values,
                "true_label": y_test.values,
                "pred_prob": test_probs,
            })

            for aggregation, agg_func in aggregation_strategies.items():
                train_task = train_base.groupby("subject_activity").agg(
                    true_label=("true_label", "first"),
                    agg_prob=("pred_prob", agg_func),
                ).reset_index()
                test_task = test_base.groupby("subject_activity").agg(
                    true_label=("true_label", "first"),
                    agg_prob=("pred_prob", agg_func),
                ).reset_index()

                threshold = tune_threshold(train_task["true_label"], train_task["agg_prob"])
                metrics = compute_metrics(test_task["true_label"], test_task["agg_prob"], threshold)

                pred_task = test_task.copy()
                pred_task["feature_set"] = feature_set
                pred_task["evaluation_stage"] = "window_to_task"
                pred_task["model"] = model_name
                pred_task["aggregation"] = aggregation
                pred_task["pca_applied"] = bool(use_pca)
                pred_task["fold"] = fold
                pred_task["threshold"] = threshold
                pred_task["prob_stress"] = pred_task["agg_prob"]
                pred_task["pred_label"] = (pred_task["prob_stress"] >= threshold).astype(int)
                pred_task["subject"] = pred_task["subject_activity"].astype(str).str.split("_", n=1).str[0]
                pred_task = pred_task.rename(columns={"true_label": "y_true"})
                prediction_rows.append(pred_task[[
                    "feature_set", "evaluation_stage", "model", "aggregation", "pca_applied",
                    "fold", "subject", "subject_activity", "y_true", "prob_stress",
                    "pred_label", "threshold",
                ]])

                fold_rows.append({
                    "feature_set": feature_set,
                    "evaluation_stage": "window_to_task",
                    "model": model_name,
                    "aggregation": aggregation,
                    "pca_applied": bool(use_pca),
                    "fold": fold,
                    "threshold": threshold,
                    "inner_best_balanced_accuracy": inner_best_score,
                    "best_params": str(best_params),
                    "n_train_tasks": train_df["subject_activity"].nunique(),
                    "n_test_tasks": test_df["subject_activity"].nunique(),
                    "n_train_subjects": len(train_subjects),
                    "n_test_subjects": len(test_subjects),
                    **metrics,
                })

    fold_df = pd.DataFrame(fold_rows)
    prediction_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
    summary = summarize_cv_results(
        fold_df,
        group_cols=["feature_set", "evaluation_stage", "model", "aggregation", "pca_applied"],
    )
    return summary, fold_df, prediction_df


AUDIO_FINAL_EVAL_CONFIGS = [
    # name, csv path, feature prefix, use_pca
    ("audeering_original", "wav2vec_audeering_embeddings_original.csv","wav2vec2_", False),
    ("audeering_original_pca", "wav2vec_audeering_embeddings_original.csv","wav2vec2_", True),
    ("librosa_original_no_vad", "audio_features_5s_0overlap_original.csv", None, False),
    ("wav2vec_original_no_vad", "wav2vec_embeddings_original.csv", "w2v_", False),
    ("wav2vec_pca_original_no_vad", "wav2vec_embeddings_original.csv", "w2v_", True),
    ("hubert_original_no_vad", "hubert_embeddings_original.csv", "hubert_", False),
    ("hubert_pca_original_no_vad", "hubert_embeddings_original.csv", "hubert_", True),
    #("ast_original_no_vad", "ast_embeddings_original.csv", "ast_", False),
    #("ast_pca_original_no_vad", "ast_embeddings_original.csv", "ast_", True),
]

all_audio_summaries = []
all_audio_folds = []
all_audio_predictions = []

for feature_set, csv_path, feature_prefix, use_pca in AUDIO_FINAL_EVAL_CONFIGS:
    if not os.path.exists(csv_path):
        print(f"Skipping {feature_set}: file not found: {csv_path}")
        continue

    print(f"\nEvaluating {feature_set} from {csv_path} | PCA={use_pca}")
    df_audio_cv = load_audio_window_table(csv_path)
    task_label_counts = df_audio_cv[["subject_activity", "y"]].drop_duplicates()["y"].value_counts().to_dict()
    print(
        f"Loaded {len(df_audio_cv)} windows | "
        f"{df_audio_cv['subject'].nunique()} subjects | "
        f"{df_audio_cv['subject_activity'].nunique()} tasks | "
        f"task labels={task_label_counts}"
    )
    feature_cols = infer_audio_feature_cols(df_audio_cv, feature_prefix=feature_prefix)
    if not feature_cols:
        print(f"Skipping {feature_set}: no feature columns found.")
        continue

    summary, folds, predictions = evaluate_audio_window_to_task_cv(
        df_audio_cv,
        feature_cols=feature_cols,
        feature_set=feature_set,
        use_pca=use_pca,
    )
    all_audio_summaries.append(summary)
    all_audio_folds.append(folds)
    all_audio_predictions.append(predictions)

if all_audio_summaries:
    audio_cv_summary = pd.concat(all_audio_summaries, ignore_index=True)
    audio_cv_summary = audio_cv_summary.sort_values(
        ["balanced_accuracy_mean", "f1_mean", "accuracy_mean"], ascending=False
    ).reset_index(drop=True)

    audio_cv_summary.to_csv("audio_original_unimodal_subject_cv_summary.csv", index=False)

    audio_cv_folds = pd.concat(all_audio_folds, ignore_index=True)
    audio_cv_folds.to_csv("audio_original_unimodal_subject_cv_folds.csv", index=False)

    audio_cv_predictions = pd.concat(all_audio_predictions, ignore_index=True)
    audio_cv_predictions.to_csv("audio_original_unimodal_subject_cv_predictions.csv", index=False)
else:
    print("No audio feature files were found for the final CV evaluation.")



Evaluating audeering_original from wav2vec_audeering_embeddings_original.csv | PCA=False
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evaluating audeering_original_pca from wav2vec_audeering_embeddings_original.csv | PCA=True
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evaluating librosa_original_no_vad from audio_features_5s_0overlap_original.csv | PCA=False
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evaluating wav2vec_original_no_vad from wav2vec_embeddings_original.csv | PCA=False
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evaluating wav2vec_pca_original_no_vad from wav2vec_embeddings_original.csv | PCA=True
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evaluating hubert_original_no_vad from hubert_embeddings_original.csv | PCA=False
Loaded 4536 windows | 54 subjects | 378 tasks | task labels={1: 268, 0: 110}

Evalu

In [3]:
audio_cv_summary.head(20)

,feature_set,evaluation_stage,model,aggregation,pca_applied,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,...,precision_mean,precision_std,recall_no_stress_mean,recall_no_stress_std,recall_stress_mean,recall_stress_std,roc_auc_mean,roc_auc_std,n_folds,n_tasks_mean
0,wav2vec_original_no_vad,window_to_task,mlp,mean,False,0.587013,0.131612,0.565711,0.078012,0.659061,...,0.756164,0.067063,0.515057,0.238982,0.616364,0.249365,0.597294,0.141393,5,75.6
1,wav2vec_pca_original_no_vad,window_to_task,xgboost,mean,True,0.556883,0.107700,0.562831,0.068899,0.623078,...,0.757589,0.063729,0.570280,0.205964,0.555381,0.204965,0.573180,0.095576,5,75.6
2,wav2vec_original_no_vad,window_to_task,balanced_rf,mean,False,0.616623,0.146904,0.560172,0.106005,0.701936,...,0.746635,0.066667,0.426369,0.220240,0.693974,0.251072,0.601401,0.153539,5,75.6
3,hubert_original_no_vad,window_to_task,mlp,mean,False,0.622597,0.102316,0.552411,0.086867,0.715651,...,0.748284,0.046596,0.389648,0.283044,0.715174,0.220721,0.565129,0.132258,5,75.6
4,wav2vec_pca_original_no_vad,window_to_task,xgboost,p75,True,0.577922,0.118851,0.550641,0.043398,0.645397,...,0.755902,0.042804,0.486542,0.319237,0.614739,0.289897,0.564528,0.065907,5,75.6
5,librosa_original_no_vad,window_to_task,random_forest,median,False,0.453766,0.075300,0.545523,0.055902,0.453487,...,0.765047,0.067950,0.763618,0.058394,0.327429,0.100798,0.501368,0.089682,5,75.6
6,wav2vec_pca_original_no_vad,window_to_task,xgboost,median,True,0.541039,0.126999,0.544608,0.092507,0.594987,...,0.762675,0.111353,0.549501,0.311026,0.539715,0.270260,0.563290,0.099576,5,75.6
7,audeering_original,window_to_task,xgboost,mean,False,0.460779,0.197151,0.543903,0.127838,0.396102,...,0.602898,0.354133,0.746264,0.242464,0.341543,0.335658,0.553815,0.165174,5,75.6
8,wav2vec_original_no_vad,window_to_task,xgboost,median,False,0.509610,0.161367,0.543754,0.063477,0.518811,...,0.789107,0.124616,0.619048,0.240375,0.468459,0.315034,0.583095,0.116354,5,75.6
9,wav2vec_original_no_vad,window_to_task,xgboost,p75,False,0.470649,0.141187,0.543346,0.093268,0.455340,...,0.606902,0.349002,0.715528,0.202627,0.371164,0.239465,0.577953,0.122753,5,75.6
